# 🔬 DISCOVERY ENGINE - ADAPT OR DIE

## Philosophy: Work Hard, Stay Ready to Pivot

**"Nothing good has been given to us. We worked for it all."**

---

### The Balance:
- **FIGHT THROUGH** when data shows promise (55%+) - don't quit at first loss
- **PIVOT FAST** when data says NO (<50%) - don't fall in love with bad ideas
- **STAY CURIOUS** - the best edge might be something we haven't tested yet

---

### What This Notebook Does:
1. **SWEEP** parameters (no hardcoded numbers - let data decide)
2. **SHOW** what's working AND what's failing (both matter)
3. **ADAPT** - easy to add new tests, change universe, try new ideas
4. **DECIDE** - clear criteria so we're not guessing

---

### Decision Framework:
| Win Rate | Action | Mindset |
|----------|--------|---------|
| **65%+** | BUILD IT NOW | This is gold, don't overthink |
| **55-65%** | REFINE IT | Worth fighting through, needs work |
| **50-55%** | INVESTIGATE | Edge or noise? Need more data |
| **<50%** | KILL IT FAST | Don't waste time, pivot to next idea |

---

### Built for Adaptation:
- Universe is a variable (swap it anytime)
- Sweeps cover wide ranges (don't assume we know the answer)
- Playground cell at the end (test ANY idea fast)
- Results show EVERYTHING (not just winners)

**We won't know until we try. Let's find out.**

In [ ]:
# CELL 1: SETUP
import pandas as pd
import numpy as np
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Check for GPU
try:
    import torch
    GPU_AVAILABLE = torch.cuda.is_available()
    if GPU_AVAILABLE:
        print(f"🔥 GPU DETECTED: {torch.cuda.get_device_name(0)}")
except:
    GPU_AVAILABLE = False
    print("⚠️ No GPU - running on CPU (still fast for this)")

print("="*70)
print("🔬 DISCOVERY ENGINE - NO HARDCODED NUMBERS")
print("="*70)
print(f"Date: {datetime.now().strftime('%Y-%m-%d %H:%M')}")
print("Mode: Parameter Sweep - Let Data Find The Edge")

In [ ]:
# CELL 2: UNIVERSE - High volatility movers that ACTUALLY move

UNIVERSE = [
    # Quantum (extreme volatility)
    'IONQ', 'RGTI', 'QMCO', 'QUBT',
    # Crypto
    'MARA', 'RIOT', 'CLSK', 'COIN',
    # Space
    'RKLB', 'ASTS', 'SPIR', 'LUNR',
    # Biotech
    'NTLA', 'BEAM', 'CRSP', 'RXRX', 'AKRO', 'VKTX',
    # AI/Tech
    'NVDA', 'AMD', 'PLTR', 'SMCI', 'AI', 'PATH',
    # EVs
    'TSLA', 'RIVN', 'LCID', 'QS', 'JOBY',
    # Clean Energy
    'PLUG', 'FCEL', 'ENPH',
    # Fintech
    'SOFI', 'UPST', 'AFRM', 'HOOD', 'SQ',
    # Movers
    'APP', 'CELH', 'DUOL'
]

print(f"Universe: {len(UNIVERSE)} tickers")
print("These are high-beta stocks that actually MOVE")

In [ ]:
# CELL 3: DATA FETCHER WITH CACHING

DATA_CACHE = {}

def get_data(ticker, days=365):
    """Fetch data with caching to avoid repeated API calls"""
    cache_key = f"{ticker}_{days}"
    if cache_key in DATA_CACHE:
        return DATA_CACHE[cache_key]
    
    try:
        end = datetime.now()
        start = end - timedelta(days=days)
        df = yf.download(ticker, start=start, end=end, progress=False)
        if len(df) > 50:
            DATA_CACHE[cache_key] = df
            return df
    except:
        pass
    return None

# Pre-fetch all data (do this ONCE)
print("Fetching data for all tickers...")
for ticker in UNIVERSE:
    df = get_data(ticker)
    if df is not None:
        print(f"  ✓ {ticker}: {len(df)} bars")
    else:
        print(f"  ✗ {ticker}: FAILED")
print(f"\n✅ Cached {len(DATA_CACHE)} datasets")

---
## 🔬 DISCOVERY 1: RSI THRESHOLD SWEEP

**Question:** At what RSI level does oversold actually predict a bounce?

**Old way:** "RSI < 21 works!" (Who picked 21? Why not 20 or 25?)

**New way:** Test RSI 5, 10, 15, 20, 25, 30... and SEE where the edge is

In [ ]:
# CELL 4: RSI THRESHOLD SWEEP - NO HARDCODED NUMBERS

def calculate_rsi(prices, period=14):
    """Calculate RSI"""
    delta = prices.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / (avg_loss + 1e-10)
    return 100 - (100 / (1 + rs))

def sweep_rsi_threshold():
    """
    SWEEP all RSI thresholds from 5 to 50
    For each: calculate win rate for bounce within N days
    """
    print("\n" + "="*70)
    print("🔬 RSI THRESHOLD SWEEP")
    print("="*70)
    
    # Parameters to sweep
    rsi_thresholds = range(10, 45, 5)  # 10, 15, 20, 25, 30, 35, 40
    hold_days_options = [3, 5, 7, 10]  # How long to hold
    target_gains = [3, 5, 8, 10]       # What counts as "win"
    
    results = []
    
    for ticker in UNIVERSE:
        df = get_data(ticker)
        if df is None or len(df) < 60:
            continue
        
        close = df['Close']
        rsi = calculate_rsi(close)
        
        for rsi_thresh in rsi_thresholds:
            for hold_days in hold_days_options:
                for target in target_gains:
                    # Find all signals
                    wins = 0
                    total = 0
                    
                    for i in range(30, len(df) - hold_days - 1):
                        if rsi.iloc[i] < rsi_thresh:
                            entry = close.iloc[i]
                            # Check max gain in hold period
                            future_prices = close.iloc[i+1:i+hold_days+1]
                            if len(future_prices) > 0:
                                max_gain = (future_prices.max() / entry - 1) * 100
                                total += 1
                                if max_gain >= target:
                                    wins += 1
                    
                    if total >= 5:  # Need minimum signals
                        results.append({
                            'ticker': ticker,
                            'rsi_threshold': rsi_thresh,
                            'hold_days': hold_days,
                            'target_gain': target,
                            'signals': total,
                            'wins': wins,
                            'win_rate': wins / total * 100
                        })
    
    return pd.DataFrame(results)

rsi_results = sweep_rsi_threshold()
print(f"\n📊 Generated {len(rsi_results)} data points")

In [ ]:
# CELL 5: ANALYZE RSI SWEEP - WHERE IS THE EDGE?

if len(rsi_results) > 0:
    # Group by parameters (ignore ticker) - find UNIVERSAL patterns
    grouped = rsi_results.groupby(['rsi_threshold', 'hold_days', 'target_gain']).agg({
        'signals': 'sum',
        'wins': 'sum',
        'win_rate': 'mean'
    }).reset_index()
    
    # Calculate actual win rate from totals
    grouped['actual_win_rate'] = grouped['wins'] / grouped['signals'] * 100
    
    # Sort by win rate
    grouped = grouped.sort_values('actual_win_rate', ascending=False)
    
    print("\n" + "="*70)
    print("🏆 TOP RSI CONFIGURATIONS (by actual win rate)")
    print("="*70)
    print(f"{'RSI<':<8} {'Hold':<8} {'Target':<10} {'Signals':<10} {'Win Rate':<10} {'Status'}")
    print("-"*70)
    
    for _, row in grouped.head(15).iterrows():
        status = '🔥 WINNER' if row['actual_win_rate'] >= 65 else '✅ GOOD' if row['actual_win_rate'] >= 55 else '❌'
        print(f"{row['rsi_threshold']:<8} {row['hold_days']:<8} {row['target_gain']}%{'':<7} {int(row['signals']):<10} {row['actual_win_rate']:.1f}%{'':<5} {status}")
    
    # Summary
    best = grouped.iloc[0]
    print("\n" + "="*70)
    print(f"💡 DISCOVERY: Best RSI config is RSI < {best['rsi_threshold']}, hold {best['hold_days']} days, target {best['target_gain']}%")
    print(f"   Win Rate: {best['actual_win_rate']:.1f}% across {int(best['signals'])} signals")
    print("="*70)
else:
    print("❌ No results - check data")

---
## 🔬 DISCOVERY 2: VOLUME SPIKE SWEEP

**Question:** Does volume actually predict price? At what ratio?

**Sweep:** Volume ratios from 1.5x to 5x average

In [ ]:
# CELL 6: VOLUME SPIKE SWEEP

def sweep_volume_threshold():
    """
    SWEEP: At what volume spike does price follow?
    Test volume ratios from 1.5x to 5x
    """
    print("\n" + "="*70)
    print("🔬 VOLUME SPIKE SWEEP")
    print("="*70)
    
    vol_ratios = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0]
    hold_days_options = [1, 2, 3, 5]
    target_gains = [2, 3, 5, 7]
    
    results = []
    
    for ticker in UNIVERSE:
        df = get_data(ticker)
        if df is None or len(df) < 60:
            continue
        
        close = df['Close'].values
        volume = df['Volume'].values
        vol_ma = pd.Series(volume).rolling(20).mean().values
        
        for vol_ratio in vol_ratios:
            for hold_days in hold_days_options:
                for target in target_gains:
                    wins = 0
                    total = 0
                    
                    for i in range(25, len(df) - hold_days - 1):
                        if vol_ma[i] > 0:
                            ratio = volume[i] / vol_ma[i]
                            price_change = abs((close[i] / close[i-1] - 1) * 100)
                            
                            # Signal: Volume spike but price hasn't moved much yet
                            if ratio >= vol_ratio and price_change < 3:
                                # Check future
                                future_max = max(close[i+1:i+hold_days+1])
                                max_gain = (future_max / close[i] - 1) * 100
                                total += 1
                                if max_gain >= target:
                                    wins += 1
                    
                    if total >= 3:
                        results.append({
                            'ticker': ticker,
                            'vol_ratio': vol_ratio,
                            'hold_days': hold_days,
                            'target_gain': target,
                            'signals': total,
                            'wins': wins,
                            'win_rate': wins / total * 100
                        })
    
    return pd.DataFrame(results)

vol_results = sweep_volume_threshold()
print(f"\n📊 Generated {len(vol_results)} data points")

In [ ]:
# CELL 7: ANALYZE VOLUME SWEEP

if len(vol_results) > 0:
    grouped = vol_results.groupby(['vol_ratio', 'hold_days', 'target_gain']).agg({
        'signals': 'sum',
        'wins': 'sum'
    }).reset_index()
    
    grouped['actual_win_rate'] = grouped['wins'] / grouped['signals'] * 100
    grouped = grouped.sort_values('actual_win_rate', ascending=False)
    
    print("\n" + "="*70)
    print("🏆 TOP VOLUME CONFIGURATIONS")
    print("="*70)
    print(f"{'Vol>Nx':<10} {'Hold':<8} {'Target':<10} {'Signals':<10} {'Win Rate':<10} {'Status'}")
    print("-"*70)
    
    for _, row in grouped.head(15).iterrows():
        status = '🔥 WINNER' if row['actual_win_rate'] >= 65 else '✅ GOOD' if row['actual_win_rate'] >= 55 else '❌'
        print(f"{row['vol_ratio']}x{'':<6} {row['hold_days']:<8} {row['target_gain']}%{'':<7} {int(row['signals']):<10} {row['actual_win_rate']:.1f}%{'':<5} {status}")
    
    best = grouped.iloc[0]
    print("\n" + "="*70)
    print(f"💡 DISCOVERY: Best volume config is {best['vol_ratio']}x spike, hold {best['hold_days']} days, target {best['target_gain']}%")
    print(f"   Win Rate: {best['actual_win_rate']:.1f}% across {int(best['signals'])} signals")
    print("="*70)
else:
    print("❌ No results")

---
## 🔬 DISCOVERY 3: CONSECUTIVE DOWN DAYS SWEEP

**Question:** After how many down days does bounce probability increase?

**Sweep:** 2, 3, 4, 5 consecutive down days

In [ ]:
# CELL 8: CONSECUTIVE DOWN DAYS SWEEP

def sweep_down_days():
    """
    SWEEP: How many consecutive down days predict a bounce?
    """
    print("\n" + "="*70)
    print("🔬 CONSECUTIVE DOWN DAYS SWEEP")
    print("="*70)
    
    down_days_options = [2, 3, 4, 5]
    min_drops = [3, 5, 7, 10]  # Minimum total % drop
    hold_days_options = [1, 2, 3, 5]
    target_gains = [2, 3, 5]
    
    results = []
    
    for ticker in UNIVERSE:
        df = get_data(ticker)
        if df is None or len(df) < 60:
            continue
        
        close = df['Close'].values
        
        for n_down in down_days_options:
            for min_drop in min_drops:
                for hold_days in hold_days_options:
                    for target in target_gains:
                        wins = 0
                        total = 0
                        
                        for i in range(n_down + 1, len(df) - hold_days - 1):
                            # Check for n consecutive down days
                            all_down = all(close[i-j] < close[i-j-1] for j in range(n_down))
                            total_drop = (close[i] / close[i-n_down] - 1) * 100
                            
                            if all_down and total_drop <= -min_drop:
                                # Check bounce
                                future_max = max(close[i+1:i+hold_days+1])
                                max_gain = (future_max / close[i] - 1) * 100
                                total += 1
                                if max_gain >= target:
                                    wins += 1
                        
                        if total >= 3:
                            results.append({
                                'ticker': ticker,
                                'down_days': n_down,
                                'min_drop': min_drop,
                                'hold_days': hold_days,
                                'target_gain': target,
                                'signals': total,
                                'wins': wins,
                                'win_rate': wins / total * 100
                            })
    
    return pd.DataFrame(results)

down_results = sweep_down_days()
print(f"\n📊 Generated {len(down_results)} data points")

In [ ]:
# CELL 9: ANALYZE DOWN DAYS SWEEP

if len(down_results) > 0:
    grouped = down_results.groupby(['down_days', 'min_drop', 'hold_days', 'target_gain']).agg({
        'signals': 'sum',
        'wins': 'sum'
    }).reset_index()
    
    grouped['actual_win_rate'] = grouped['wins'] / grouped['signals'] * 100
    grouped = grouped.sort_values('actual_win_rate', ascending=False)
    
    print("\n" + "="*70)
    print("🏆 TOP DOWN DAYS CONFIGURATIONS")
    print("="*70)
    print(f"{'Days':<6} {'Drop>':<8} {'Hold':<6} {'Target':<8} {'Signals':<10} {'Win Rate':<10} {'Status'}")
    print("-"*70)
    
    for _, row in grouped.head(15).iterrows():
        status = '🔥 WINNER' if row['actual_win_rate'] >= 65 else '✅ GOOD' if row['actual_win_rate'] >= 55 else '❌'
        print(f"{row['down_days']:<6} {row['min_drop']}%{'':<5} {row['hold_days']:<6} {row['target_gain']}%{'':<5} {int(row['signals']):<10} {row['actual_win_rate']:.1f}%{'':<5} {status}")
    
    best = grouped.iloc[0]
    print("\n" + "="*70)
    print(f"💡 DISCOVERY: Best config is {best['down_days']} down days, >{best['min_drop']}% drop, hold {best['hold_days']} days")
    print(f"   Win Rate: {best['actual_win_rate']:.1f}% across {int(best['signals'])} signals")
    print("="*70)
else:
    print("❌ No results")

---
## 🔬 DISCOVERY 4: GAP FADE SWEEP

**Question:** Do gap-ups fade? At what gap size?

**Sweep:** Gap sizes from 3% to 15%

In [ ]:
# CELL 10: GAP FADE SWEEP

def sweep_gap_fade():
    """
    SWEEP: At what gap size do we see reliable fades?
    """
    print("\n" + "="*70)
    print("🔬 GAP FADE SWEEP")
    print("="*70)
    
    gap_sizes = [3, 5, 7, 10, 15]
    fade_targets = [2, 3, 5]  # How much it needs to fade
    
    results = []
    
    for ticker in UNIVERSE:
        df = get_data(ticker)
        if df is None or len(df) < 60:
            continue
        
        for gap_min in gap_sizes:
            for fade_target in fade_targets:
                wins = 0
                total = 0
                
                for i in range(2, len(df) - 1):
                    # Gap up
                    gap = (df['Open'].iloc[i] / df['Close'].iloc[i-1] - 1) * 100
                    
                    if gap >= gap_min:
                        # Check fade (high to close)
                        high = df['High'].iloc[i]
                        close = df['Close'].iloc[i]
                        fade = (high - close) / high * 100
                        
                        total += 1
                        if fade >= fade_target:
                            wins += 1
                
                if total >= 3:
                    results.append({
                        'ticker': ticker,
                        'gap_min': gap_min,
                        'fade_target': fade_target,
                        'signals': total,
                        'wins': wins,
                        'win_rate': wins / total * 100
                    })
    
    return pd.DataFrame(results)

gap_results = sweep_gap_fade()
print(f"\n📊 Generated {len(gap_results)} data points")

In [ ]:
# CELL 11: ANALYZE GAP FADE SWEEP

if len(gap_results) > 0:
    grouped = gap_results.groupby(['gap_min', 'fade_target']).agg({
        'signals': 'sum',
        'wins': 'sum'
    }).reset_index()
    
    grouped['actual_win_rate'] = grouped['wins'] / grouped['signals'] * 100
    grouped = grouped.sort_values('actual_win_rate', ascending=False)
    
    print("\n" + "="*70)
    print("🏆 TOP GAP FADE CONFIGURATIONS")
    print("="*70)
    print(f"{'Gap>':<10} {'Fade>':<10} {'Signals':<10} {'Win Rate':<10} {'Status'}")
    print("-"*70)
    
    for _, row in grouped.iterrows():
        status = '🔥 WINNER' if row['actual_win_rate'] >= 65 else '✅ GOOD' if row['actual_win_rate'] >= 55 else '❌'
        print(f"{row['gap_min']}%{'':<7} {row['fade_target']}%{'':<7} {int(row['signals']):<10} {row['actual_win_rate']:.1f}%{'':<5} {status}")
    
    best = grouped.iloc[0]
    print("\n" + "="*70)
    print(f"💡 DISCOVERY: Best gap fade is gap >{best['gap_min']}%, expect >{best['fade_target']}% fade")
    print(f"   Win Rate: {best['actual_win_rate']:.1f}% across {int(best['signals'])} signals")
    print("="*70)
else:
    print("❌ No results")

---
## 📊 FINAL DISCOVERY SUMMARY

This cell brings together ALL discoveries and shows what the DATA found

In [ ]:
# CELL 12: FINAL SUMMARY - WHAT DID THE DATA FIND?

print("\n" + "="*70)
print("📊 FINAL DISCOVERY SUMMARY")
print("="*70)
print("\nThe DATA found these patterns (no human bias):")
print("-"*70)

discoveries = []

# RSI Discovery
if len(rsi_results) > 0:
    g = rsi_results.groupby(['rsi_threshold', 'hold_days', 'target_gain']).agg({'signals': 'sum', 'wins': 'sum'}).reset_index()
    g['wr'] = g['wins'] / g['signals'] * 100
    best = g.loc[g['wr'].idxmax()]
    discoveries.append(('RSI Oversold', f"RSI < {best['rsi_threshold']}, hold {best['hold_days']}d, target {best['target_gain']}%", best['wr'], int(best['signals'])))

# Volume Discovery
if len(vol_results) > 0:
    g = vol_results.groupby(['vol_ratio', 'hold_days', 'target_gain']).agg({'signals': 'sum', 'wins': 'sum'}).reset_index()
    g['wr'] = g['wins'] / g['signals'] * 100
    best = g.loc[g['wr'].idxmax()]
    discoveries.append(('Volume Spike', f"Vol > {best['vol_ratio']}x, hold {best['hold_days']}d, target {best['target_gain']}%", best['wr'], int(best['signals'])))

# Down Days Discovery
if len(down_results) > 0:
    g = down_results.groupby(['down_days', 'min_drop', 'hold_days', 'target_gain']).agg({'signals': 'sum', 'wins': 'sum'}).reset_index()
    g['wr'] = g['wins'] / g['signals'] * 100
    best = g.loc[g['wr'].idxmax()]
    discoveries.append(('Down Days', f"{best['down_days']} days, >{best['min_drop']}% drop, hold {best['hold_days']}d", best['wr'], int(best['signals'])))

# Gap Fade Discovery
if len(gap_results) > 0:
    g = gap_results.groupby(['gap_min', 'fade_target']).agg({'signals': 'sum', 'wins': 'sum'}).reset_index()
    g['wr'] = g['wins'] / g['signals'] * 100
    best = g.loc[g['wr'].idxmax()]
    discoveries.append(('Gap Fade', f"Gap > {best['gap_min']}%, fade > {best['fade_target']}%", best['wr'], int(best['signals'])))

# Print discoveries
print(f"\n{'PATTERN':<18} {'BEST CONFIG':<45} {'WIN RATE':<12} {'SIGNALS':<10} {'STATUS'}")
print("-"*100)

winners = []
for name, config, wr, signals in sorted(discoveries, key=lambda x: -x[2]):
    status = '🔥 WINNER' if wr >= 65 else '✅ GOOD' if wr >= 55 else '❌ FAIL'
    print(f"{name:<18} {config:<45} {wr:.1f}%{'':<7} {signals:<10} {status}")
    if wr >= 65:
        winners.append((name, config, wr))

print("\n" + "="*70)
if len(winners) > 0:
    print("🏆 WINNERS (65%+ win rate):")
    for name, config, wr in winners:
        print(f"   • {name}: {config} ({wr:.1f}%)")
    print("\n🎯 NEXT STEP: Build live scanner for these patterns!")
else:
    print("❌ No patterns above 65% - but check the 55%+ ones")
    print("\n🎯 NEXT STEP: Refine promising patterns or test new hypotheses")
print("="*70)

---
## 🧪 PLAYGROUND: Test Your Own Hypothesis

Use this cell to quickly test any idea without hardcoding

In [ ]:
# CELL 13: PLAYGROUND - Quick hypothesis testing

def test_hypothesis(condition_func, description, hold_days=3, target_gain=5):
    """
    Generic hypothesis tester.
    condition_func(df, i) -> True if signal at index i
    """
    print(f"\n🧪 Testing: {description}")
    print("-"*50)
    
    all_signals = []
    
    for ticker in UNIVERSE:
        df = get_data(ticker)
        if df is None or len(df) < 60:
            continue
        
        close = df['Close'].values
        
        for i in range(30, len(df) - hold_days - 1):
            if condition_func(df, i):
                future_max = max(close[i+1:i+hold_days+1])
                gain = (future_max / close[i] - 1) * 100
                all_signals.append(gain >= target_gain)
    
    if len(all_signals) > 0:
        win_rate = sum(all_signals) / len(all_signals) * 100
        status = '🔥 WINNER' if win_rate >= 65 else '✅ GOOD' if win_rate >= 55 else '❌ FAIL'
        print(f"Signals: {len(all_signals)}")
        print(f"Win Rate: {win_rate:.1f}% {status}")
        return win_rate
    else:
        print("No signals found")
        return 0

# EXAMPLE: Test RSI < 25 AND Volume > 1.5x average
def combo_signal(df, i):
    close = df['Close']
    volume = df['Volume'].values
    vol_ma = pd.Series(volume).rolling(20).mean().values
    
    # Calculate RSI
    rsi = calculate_rsi(close)
    
    if i < 25 or i >= len(rsi) or vol_ma[i] == 0:
        return False
    
    return rsi.iloc[i] < 25 and volume[i] / vol_ma[i] > 1.5

# Run it
test_hypothesis(combo_signal, "RSI < 25 AND Volume > 1.5x", hold_days=5, target_gain=5)

---
## 🔄 PIVOT PLAN: What If Everything Fails?

**If ALL patterns are <55%:**
1. **Don't panic** - now we KNOW these don't work (that's valuable)
2. **Check the data** - are we testing on the right tickers? Right timeframe?
3. **Flip the hypothesis** - if RSI oversold doesn't predict bounce, does RSI OVERBOUGHT predict fade?
4. **Combine signals** - single indicators might be weak, combinations might be strong
5. **Change the universe** - maybe these patterns work on different stocks

**If SOME patterns are 55-65%:**
1. **Refine thresholds** - narrow the sweep, find the sweet spot
2. **Add filters** - volume confirmation? Sector filter? Market regime?
3. **Test combinations** - RSI oversold + Volume spike together?
4. **More data** - extend lookback, add more tickers

**Questions to Ask:**
- Are we testing the right THING? (maybe the edge isn't in price patterns)
- Are we testing the right WAY? (maybe we need different exit rules)
- Are we testing the right UNIVERSE? (maybe we need different stocks)

**Remember:** Hard work + adaptation = unstoppable. We adjust, we don't quit.

In [ ]:
# CELL 14: COMBO TESTER - What if we combine signals?
# If individual signals are weak, maybe combinations are strong

def sweep_combos():
    """
    Test combinations of signals - RSI + Volume, Down Days + Volume, etc.
    Sometimes the edge is in the COMBINATION
    """
    print("\n" + "="*70)
    print("🔬 COMBINATION SWEEP - Finding Hidden Edges")
    print("="*70)
    
    results = []
    
    for ticker in UNIVERSE:
        df = get_data(ticker)
        if df is None or len(df) < 60:
            continue
        
        close = df['Close']
        volume = df['Volume'].values
        vol_ma = pd.Series(volume).rolling(20).mean().values
        rsi = calculate_rsi(close)
        
        # Test different combos
        combos = {
            'RSI_only': lambda i: rsi.iloc[i] < 30,
            'Vol_only': lambda i: volume[i] / vol_ma[i] > 2 if vol_ma[i] > 0 else False,
            'RSI + Vol': lambda i: rsi.iloc[i] < 30 and (volume[i] / vol_ma[i] > 1.5 if vol_ma[i] > 0 else False),
            'RSI + Down2': lambda i: rsi.iloc[i] < 30 and close.iloc[i] < close.iloc[i-1] < close.iloc[i-2],
            'Vol + Down2': lambda i: (volume[i] / vol_ma[i] > 2 if vol_ma[i] > 0 else False) and close.iloc[i] < close.iloc[i-1] < close.iloc[i-2],
            'Triple': lambda i: rsi.iloc[i] < 30 and (volume[i] / vol_ma[i] > 1.5 if vol_ma[i] > 0 else False) and close.iloc[i] < close.iloc[i-1],
        }
        
        for combo_name, condition in combos.items():
            wins = 0
            total = 0
            
            for i in range(30, len(df) - 6):
                try:
                    if condition(i):
                        entry = close.iloc[i]
                        future_max = close.iloc[i+1:i+6].max()
                        gain = (future_max / entry - 1) * 100
                        total += 1
                        if gain >= 5:
                            wins += 1
                except:
                    continue
            
            if total >= 3:
                results.append({
                    'ticker': ticker,
                    'combo': combo_name,
                    'signals': total,
                    'wins': wins,
                    'win_rate': wins / total * 100
                })
    
    if len(results) > 0:
        df_results = pd.DataFrame(results)
        grouped = df_results.groupby('combo').agg({
            'signals': 'sum',
            'wins': 'sum'
        }).reset_index()
        grouped['win_rate'] = grouped['wins'] / grouped['signals'] * 100
        grouped = grouped.sort_values('win_rate', ascending=False)
        
        print(f"\n{'COMBO':<15} {'SIGNALS':<10} {'WIN RATE':<12} {'STATUS'}")
        print("-"*50)
        for _, row in grouped.iterrows():
            status = '🔥' if row['win_rate'] >= 65 else '✅' if row['win_rate'] >= 55 else '❌'
            print(f"{row['combo']:<15} {int(row['signals']):<10} {row['win_rate']:.1f}%{'':<7} {status}")
        
        print("\n💡 If combos beat singles, that's your edge!")
        return grouped
    
    return None

# Run it
combo_results = sweep_combos()